In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# 分解后的子问题分别回答

## 阶段一：将问题分解为子问题

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# 问题分解
template = """你是一个能根据输入问题生成多个子问题的有用助手。
目标是将输入分解为一组可以独立回答的子问题。
生成与原始问题相关的 3 个子问题。
原始问题：{question}。
输出 3 个可以独立回答的子问题："""

prompt_decomposition = ChatPromptTemplate.from_messages(
    [
        ("system", template)
    ]
)

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = ChatOpenAI(temperature=0)

# Output Parser
output_parser = StrOutputParser()

# Chain
generate_querires_decomposition = prompt_decomposition | llm | output_parser | (lambda x: x.strip().split("\n"))

In [4]:
question = "zero shot 与 few shot 的区别是什么？"

In [5]:
# Run
sub_questions = generate_querires_decomposition.invoke({"question": question})

sub_questions

['1. zero shot 学习方法的原理是什么？',
 '2. few shot 学习方法如何应用于机器学习任务中？',
 '3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？']

## 阶段二：逐个解决子问题

伪代码：

```python
sub_result = []

for sub_question in sub_questions:
    # 打印子问题
    print(f"Q: {sub_question}")
    # 获取子问题答案
    sub_answer = RAG(sub_question)
    # 打印子问题答案
    print(f"A: {sub_answer}\n")
    # 存储问题-答案对
    sub_result.append((sub_question, sub_answer))
    
sub_result
```

### 索引

In [6]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

### 检索与生成

In [7]:
# Generate

# Prompt
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

# # Run
# question = "什么是few shot？"
# print(f"Q: {question}\nA: ")

# result = rag_chain.invoke(question)
# print(result)

### 获取子问题的问答对

In [8]:
sub_result = []

for sub_question in sub_questions:
    # 打印子问题
    print(f"Q: {sub_question}")
    # 获取子问题答案
    sub_answer = rag_chain.invoke(sub_question)
    # 打印子问题答案
    print(f"A: {sub_answer}\n")
    # 存储问题-答案对
    sub_result.append((sub_question, sub_answer))
    
sub_result

Q: 1. zero shot 学习方法的原理是什么？
A: 零样本学习方法的原理是直接将任务文本输入模型并询问结果，而无需提供任何示例数据。模型通过从输入文本中推断出正确的答案，而不是依赖于先前的训练数据。

Q: 2. few shot 学习方法如何应用于机器学习任务中？
A: 少样本学习方法通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而在目标任务上取得更好的性能。这些示例包含目标任务的输入和期望输出，有助于模型进行泛化。

少样本学习在机器学习任务中可以通过提供少量的训练示例来帮助模型学习新任务，而无需大量的标记数据。这种方法可以提高模型的泛化能力，并且有助于解决数据稀缺性的问题。

Q: 3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？
A: 零样本学习方法是直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解人类的意图和评价标准。零样本学习通常比少样本学习更节省资源，适用于对模型训练样本数量有限的场景，但性能可能不如少样本学习方法稳定。

零样本学习方法适用于对模型训练样本数量有限的场景，而少样本学习方法在需要更多示例帮助模型理解任务时表现更好。零样本方法直接输入任务文本，而少样本方法提供示例输入和输出帮助模型理解人类意图。



[('1. zero shot 学习方法的原理是什么？',
  '零样本学习方法的原理是直接将任务文本输入模型并询问结果，而无需提供任何示例数据。模型通过从输入文本中推断出正确的答案，而不是依赖于先前的训练数据。'),
 ('2. few shot 学习方法如何应用于机器学习任务中？',
  '少样本学习方法通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而在目标任务上取得更好的性能。这些示例包含目标任务的输入和期望输出，有助于模型进行泛化。\n\n少样本学习在机器学习任务中可以通过提供少量的训练示例来帮助模型学习新任务，而无需大量的标记数据。这种方法可以提高模型的泛化能力，并且有助于解决数据稀缺性的问题。'),
 ('3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？',
  '零样本学习方法是直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解人类的意图和评价标准。零样本学习通常比少样本学习更节省资源，适用于对模型训练样本数量有限的场景，但性能可能不如少样本学习方法稳定。\n\n零样本学习方法适用于对模型训练样本数量有限的场景，而少样本学习方法在需要更多示例帮助模型理解任务时表现更好。零样本方法直接输入任务文本，而少样本方法提供示例输入和输出帮助模型理解人类意图。')]

### 拼接子问题的问答对

In [9]:
def format_qa_pair(question, answer):
    """格式化问答对"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

In [13]:
context = ""

for (sub_question, sub_answer) in sub_result:
    context += format_qa_pair(sub_question, sub_answer) + "\n"

print(context)

Question: 1. zero shot 学习方法的原理是什么？
Answer: 零样本学习方法的原理是直接将任务文本输入模型并询问结果，而无需提供任何示例数据。模型通过从输入文本中推断出正确的答案，而不是依赖于先前的训练数据。
Question: 2. few shot 学习方法如何应用于机器学习任务中？
Answer: 少样本学习方法通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而在目标任务上取得更好的性能。这些示例包含目标任务的输入和期望输出，有助于模型进行泛化。

少样本学习在机器学习任务中可以通过提供少量的训练示例来帮助模型学习新任务，而无需大量的标记数据。这种方法可以提高模型的泛化能力，并且有助于解决数据稀缺性的问题。
Question: 3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？
Answer: 零样本学习方法是直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解人类的意图和评价标准。零样本学习通常比少样本学习更节省资源，适用于对模型训练样本数量有限的场景，但性能可能不如少样本学习方法稳定。

零样本学习方法适用于对模型训练样本数量有限的场景，而少样本学习方法在需要更多示例帮助模型理解任务时表现更好。零样本方法直接输入任务文本，而少样本方法提供示例输入和输出帮助模型理解人类意图。



### 最终的chain & 生成

In [11]:
# Prompt
system_prompt = (
    "你是一个用于问答任务的助手。"
    "基于以下对原始问题分解后得到的子问题的问答对来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

# Chain
final_chain = prompt | llm | output_parser

In [15]:
question

'zero shot 与 few shot 的区别是什么？'

In [16]:
context

'Question: 1. zero shot 学习方法的原理是什么？\nAnswer: 零样本学习方法的原理是直接将任务文本输入模型并询问结果，而无需提供任何示例数据。模型通过从输入文本中推断出正确的答案，而不是依赖于先前的训练数据。\nQuestion: 2. few shot 学习方法如何应用于机器学习任务中？\nAnswer: 少样本学习方法通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而在目标任务上取得更好的性能。这些示例包含目标任务的输入和期望输出，有助于模型进行泛化。\n\n少样本学习在机器学习任务中可以通过提供少量的训练示例来帮助模型学习新任务，而无需大量的标记数据。这种方法可以提高模型的泛化能力，并且有助于解决数据稀缺性的问题。\nQuestion: 3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？\nAnswer: 零样本学习方法是直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解人类的意图和评价标准。零样本学习通常比少样本学习更节省资源，适用于对模型训练样本数量有限的场景，但性能可能不如少样本学习方法稳定。\n\n零样本学习方法适用于对模型训练样本数量有限的场景，而少样本学习方法在需要更多示例帮助模型理解任务时表现更好。零样本方法直接输入任务文本，而少样本方法提供示例输入和输出帮助模型理解人类意图。\n'

In [17]:
final_chain.invoke({"context":context, "question":question})

'零样本学习方法直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解任务。零样本学习通常比少样本学习更节省资源，但性能可能不如少样本学习方法稳定。零样本适用于样本数量有限的场景，而少样本学习在需要更多示例帮助模型理解任务时表现更好。'

In [18]:
prompt.invoke({"context":context, "question":question})

ChatPromptValue(messages=[SystemMessage(content='你是一个用于问答任务的助手。基于以下对原始问题分解后得到的子问题的问答对来回答问题。如果你不知道答案，就说你不知道。答案最多使用三句话，并保持简洁。\n\nQuestion: 1. zero shot 学习方法的原理是什么？\nAnswer: 零样本学习方法的原理是直接将任务文本输入模型并询问结果，而无需提供任何示例数据。模型通过从输入文本中推断出正确的答案，而不是依赖于先前的训练数据。\nQuestion: 2. few shot 学习方法如何应用于机器学习任务中？\nAnswer: 少样本学习方法通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而在目标任务上取得更好的性能。这些示例包含目标任务的输入和期望输出，有助于模型进行泛化。\n\n少样本学习在机器学习任务中可以通过提供少量的训练示例来帮助模型学习新任务，而无需大量的标记数据。这种方法可以提高模型的泛化能力，并且有助于解决数据稀缺性的问题。\nQuestion: 3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？\nAnswer: 零样本学习方法是直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解人类的意图和评价标准。零样本学习通常比少样本学习更节省资源，适用于对模型训练样本数量有限的场景，但性能可能不如少样本学习方法稳定。\n\n零样本学习方法适用于对模型训练样本数量有限的场景，而少样本学习方法在需要更多示例帮助模型理解任务时表现更好。零样本方法直接输入任务文本，而少样本方法提供示例输入和输出帮助模型理解人类意图。\n'), HumanMessage(content='zero shot 与 few shot 的区别是什么？')])

In [19]:
print(prompt.invoke({"context":context, "question":question}).messages[0].content)

你是一个用于问答任务的助手。基于以下对原始问题分解后得到的子问题的问答对来回答问题。如果你不知道答案，就说你不知道。答案最多使用三句话，并保持简洁。

Question: 1. zero shot 学习方法的原理是什么？
Answer: 零样本学习方法的原理是直接将任务文本输入模型并询问结果，而无需提供任何示例数据。模型通过从输入文本中推断出正确的答案，而不是依赖于先前的训练数据。
Question: 2. few shot 学习方法如何应用于机器学习任务中？
Answer: 少样本学习方法通过提供一系列高质量的示例，帮助模型更好地理解人类的意图和评价标准，从而在目标任务上取得更好的性能。这些示例包含目标任务的输入和期望输出，有助于模型进行泛化。

少样本学习在机器学习任务中可以通过提供少量的训练示例来帮助模型学习新任务，而无需大量的标记数据。这种方法可以提高模型的泛化能力，并且有助于解决数据稀缺性的问题。
Question: 3. zero shot 和 few shot 学习方法在性能和应用场景上有何区别？
Answer: 零样本学习方法是直接将任务文本输入模型并询问结果，而少样本学习方法通过提供一系列高质量的示例帮助模型更好地理解人类的意图和评价标准。零样本学习通常比少样本学习更节省资源，适用于对模型训练样本数量有限的场景，但性能可能不如少样本学习方法稳定。

零样本学习方法适用于对模型训练样本数量有限的场景，而少样本学习方法在需要更多示例帮助模型理解任务时表现更好。零样本方法直接输入任务文本，而少样本方法提供示例输入和输出帮助模型理解人类意图。

